In [1]:
from pathlib import Path
import datetime as dt
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path(r"D:\Driveguard")
df = pl.read_parquet(ROOT / "data" / "features_ST12000NM0008.parquet")

SPLIT = dt.date(2026, 4, 1)      # test = Q2 2026 (newest data, never seen in training)
GAP = dt.timedelta(days=30)      # embargo: train labels must not peek into the test period

train = df.filter(pl.col("date") < SPLIT - GAP)
test  = df.filter(pl.col("date") >= SPLIT)

DROP = ["serial_number", "date", "model", "capacity_bytes", "label", "days_to_failure",
        "smart_9_raw", "smart_194_raw"]
FEATURES = [c for c in df.columns if c not in DROP]

print("Train:", train.shape, "positives:", train["label"].sum())
print("Test: ", test.shape,  "positives:", test["label"].sum())
print(len(FEATURES), "features")

Train: (4555363, 37) positives: 8790
Test:  (1122154, 37) positives: 2915
29 features


In [2]:
pos = train.filter(pl.col("label") == 1)
neg = train.filter(pl.col("label") == 0).sample(fraction=0.10, seed=42)
train_s = pl.concat([pos, neg]).sample(fraction=1.0, shuffle=True, seed=42)

X_train = train_s.select(FEATURES).to_pandas()
y_train = train_s["label"].to_numpy()
X_test  = test.select(FEATURES).to_pandas()
y_test  = test["label"].to_numpy()
print("Training rows:", len(X_train), "| failing share:", round(y_train.mean() * 100, 2), "%")

Training rows: 463447 | failing share: 1.9 %


In [3]:
model = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, random_state=42, verbose=-1,
)
model.fit(X_train, y_train)

test = test.with_columns(
    pl.Series("score", model.predict_proba(X_test)[:, 1]),
    pl.max_horizontal([pl.col(c) > 0 for c in
        ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw", "smart_198_raw"]])
      .cast(pl.Int8).alias("rule_flag"),
)

print("PR-AUC  | model:", round(average_precision_score(y_test, test["score"]), 3),
      "| rule:", round(average_precision_score(y_test, test["rule_flag"]), 3),
      "| random guess:", round(y_test.mean(), 4))
print("ROC-AUC | model:", round(roc_auc_score(y_test, test["score"]), 3),
      "| rule:", round(roc_auc_score(y_test, test["rule_flag"]), 3))

PR-AUC  | model: 0.292 | rule: 0.012 | random guess: 0.0026
ROC-AUC | model: 0.866 | rule: 0.847


In [4]:
drives = test.group_by("serial_number").agg(
    pl.col("label").max().alias("failed"),
    pl.col("rule_flag").max().alias("rule_alert"),
    pl.col("score").max().alias("max_score"),
)
n_fail = drives["failed"].sum()
n_ok = drives.height - n_fail

def drive_metrics(alert_col):
    caught = drives.filter((pl.col("failed") == 1) & (pl.col(alert_col) == 1)).height
    false_alarms = drives.filter((pl.col("failed") == 0) & (pl.col(alert_col) == 1)).height
    return caught, false_alarms

r_caught, r_fa = drive_metrics("rule_alert")

# model threshold that raises the SAME number of false alarms as the rule
ok_scores = np.sort(drives.filter(pl.col("failed") == 0)["max_score"].to_numpy())[::-1]
thr_same = ok_scores[r_fa] if r_fa < len(ok_scores) else 0.0
drives = drives.with_columns((pl.col("max_score") > thr_same).cast(pl.Int8).alias("model_alert"))
m_caught, m_fa = drive_metrics("model_alert")

print(f"Test quarter: {drives.height:,} drives | {n_fail} failed")
print(f"Rule  (any SMART error > 0): caught {r_caught}/{n_fail} = {r_caught/n_fail:.1%} | false alarms {r_fa:,} ({r_fa/n_ok:.1%} of healthy drives)")
print(f"Model (same false alarms)  : caught {m_caught}/{n_fail} = {m_caught/n_fail:.1%} | false alarms {m_fa:,}")

Test quarter: 18,521 drives | 145 failed
Rule  (any SMART error > 0): caught 134/145 = 92.4% | false alarms 3,128 (17.0% of healthy drives)
Model (same false alarms)  : caught 130/145 = 89.7% | false alarms 3,123


In [5]:
rows = []
for fa_rate in [0.005, 0.01, 0.02, 0.05]:
    k = int(fa_rate * n_ok)
    thr = ok_scores[k] if k < len(ok_scores) else 0.0
    caught = drives.filter((pl.col("failed") == 1) & (pl.col("max_score") > thr)).height
    rows.append({"false_alarm_rate": f"{fa_rate:.1%}", "false_alarm_drives": k,
                 "failures_caught": caught, "recall": f"{caught/n_fail:.1%}",
                 "threshold": round(float(thr), 4)})
pd.DataFrame(rows)

,false_alarm_rate,false_alarm_drives,failures_caught,recall,threshold
0,0.5%,91,111,76.6%,0.5509
1,1.0%,183,122,84.1%,0.2951
2,2.0%,367,124,85.5%,0.1487
3,5.0%,918,126,86.9%,0.0591


In [6]:
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump({"model": model, "features": FEATURES, "threshold": thr_same},
            ROOT / "models" / "lgbm_ST12000NM0008.joblib")

imp = pd.DataFrame({"feature": FEATURES,
                    "importance": model.booster_.feature_importance("gain")})
imp.sort_values("importance", ascending=False).head(10)

,feature,importance
9,smart_187_diff7,118365.235072
10,smart_187_diff30,60135.513524
11,smart_187_max30,59847.759722
25,age_days,45084.277283
1,smart_187_raw,38705.792604
27,temp_mean7,34464.367500
12,smart_187_days_nonzero30,31993.666981
14,smart_188_diff30,31487.625332
0,smart_5_raw,19171.423965
7,smart_5_max30,15483.557624
